In [ ]:
import zipfile
from pathlib import Path

import cdsapi

# Only 25 August 2026 is needed for the daily-mean wind map
output_folder = Path("Output/wind_analysis")
output_folder.mkdir(parents=True, exist_ok=True)
zip_path = output_folder / "era5_wind_20260825.zip"

dataset = "reanalysis-era5-single-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": [
        "10m_u_component_of_wind",
        "10m_v_component_of_wind"
    ],
    "year": ["2026"],
    "month": ["08"],
    "day": ["25"],
    "time": [f"{hour:02d}:00" for hour in range(24)],
    "data_format": "netcdf",
    "download_format": "zip",
    "area": [46.25, -2, 43.75, 1.25]
}

# Requires your own CDS API key in ~/.cdsapirc
# (see https://cds.climate.copernicus.eu/how-to-api).
client = cdsapi.Client()
client.retrieve(dataset, request).download(str(zip_path))

# Unpack next to the zip so the next cell reads the 25 August data
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(output_folder)
    print("Extracted:", archive.namelist())

In [ ]:
import numpy as np
import xarray as xr
from pathlib import Path

hourly_file = Path("Output/wind_analysis/data_stream-oper_stepType-instant.nc")
with xr.open_dataset(hourly_file) as hourly_ds:
    daily_wind = hourly_ds[["u10", "v10"]].resample(valid_time="1D").mean().load()

daily_wind["wind_speed"] = np.hypot(daily_wind["u10"], daily_wind["v10"])
daily_wind["wind_speed"].attrs = {
    "long_name": "Daily mean 10 metre wind speed from mean components",
    "units": "m s**-1",
}
daily_wind["wind_direction"] = (
    270 - np.degrees(np.arctan2(daily_wind["v10"], daily_wind["u10"]))
) % 360
daily_wind["wind_direction"].attrs = {
    "long_name": "Daily mean 10 metre wind direction from mean components",
    "units": "degrees",
}

print(f"Hourly timesteps: {hourly_ds.sizes['valid_time']}")
print(f"Daily timesteps: {daily_wind.sizes['valid_time']}")
print("Daily averages for u10 and v10, plus derived wind speed and direction:")
display(daily_wind)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path

# Use the daily average of 25 August 2026 (labelled 2026-08-25 00:00 UTC) computed in cell 2.
target_day = "2026-08-25"
day_wind = daily_wind.sel(valid_time=target_day).squeeze()

speed = day_wind["wind_speed"].values
u = day_wind["u10"].values
v = day_wind["v10"].values
lat = daily_wind["latitude"].values
lon = daily_wind["longitude"].values
map_projection = ccrs.PlateCarree()

print(f"Wind speed on {target_day}: {np.nanmin(speed):.2f} to {np.nanmax(speed):.2f} m/s")

figure_folder = Path("Output/figures")
figure_folder.mkdir(parents=True, exist_ok=True)

cities = {
    "La Rochelle": (-1.1511, 46.1603),
    "Rochefort": (-0.9830, 45.9430),
    "Saintes": (-0.6340, 45.7460),
    "Bordeaux": (-0.5792, 44.8378),
    "Arcachon": (-1.1700, 44.6500)
}

figure, map_ax = plt.subplots(
    figsize=(12, 10),
    subplot_kw={"projection": map_projection},
    constrained_layout=True,
)
map_ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=map_projection)
map_ax.add_feature(cfeature.OCEAN, facecolor="#dbeaf2", zorder=0)
map_ax.add_feature(cfeature.LAND, facecolor="#eeeeea", zorder=0)
map_ax.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor="#444444", zorder=3)
map_ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor="#666666", zorder=3)

raster = map_ax.pcolormesh(
    lon,
    lat,
    speed,
    shading="auto",
    cmap="viridis",
    vmin=0,
    vmax=float(np.nanmax(speed)),
    alpha=0.62,
    transform=map_projection,
    zorder=1,
)
map_ax.quiver(
    lon,
    lat,
    u,
    v,
    color="black",
    pivot="mid",
    scale=70,
    width=0.003,
    alpha=0.55,
    transform=map_projection,
    zorder=2,
)

city_longitudes = [coordinates[0] for coordinates in cities.values()]
city_latitudes = [coordinates[1] for coordinates in cities.values()]
map_ax.scatter(
    city_longitudes,
    city_latitudes,
    c="red",
    s=50,
    edgecolors="white",
    linewidths=1.2,
    transform=map_projection,
    zorder=4,
)
for city_name, (city_lon, city_lat) in cities.items():
    map_ax.text(
        city_lon + 0.02,
        city_lat + 0.02,
        city_name,
        fontsize=18 if city_name == "Bordeaux" else 14,
        color="black",
        ha="left",
        va="bottom",
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.7, "pad": 2},
        transform=map_projection,
        zorder=5,
    )

map_ax.set_title(f"{target_day}", fontsize=16)
map_ax.set_xlabel("Longitude", fontsize=16)
map_ax.set_ylabel("Latitude", fontsize=16)
gridlines = map_ax.gridlines(
    draw_labels=True,
    linewidth=0.4,
    alpha=0.5,
    linestyle="--",
)
gridlines.top_labels = False
gridlines.right_labels = False
gridlines.xlabel_style = {"size": 16}
gridlines.ylabel_style = {"size": 16}

# Colorbar axes in map-axes coordinates, so it matches the map height exactly
colorbar_ax = map_ax.inset_axes([1.03, 0, 0.04, 1])
colorbar = figure.colorbar(raster, cax=colorbar_ax, orientation="vertical")
colorbar.set_label("Windsnelheid (m/s)", fontsize=22)
colorbar.ax.tick_params(labelsize=22)

output_svg = figure_folder / f"wind_{target_day.replace('-', '')}.svg"
figure.savefig(output_svg, format="svg", bbox_inches="tight")
plt.show()
plt.close(figure)
print(f"Saved {output_svg}")

In [ ]:
from IPython.display import display

# Child-friendly version of the figure above: no numbers on the colorbar,
# just "Laag" at the bottom, "Hoog" at the top and an arrow from low to high.
low_label, high_label = "Laag", "Hoog"

# Remove earlier additions so re-running this cell does not stack them
for artist in [text for text in colorbar.ax.texts if text.get_gid() == "kids_colorbar"]:
    artist.remove()

colorbar.set_ticks([])
colorbar.ax.text(0.5, -0.01, low_label, transform=colorbar.ax.transAxes,
                 ha="center", va="top", fontsize=22, gid="kids_colorbar")
colorbar.ax.text(0.5, 1.01, high_label, transform=colorbar.ax.transAxes,
                 ha="center", va="bottom", fontsize=22, gid="kids_colorbar")
colorbar.ax.annotate(
    "",
    xy=(2.0, 0.97),
    xytext=(2.0, 0.03),
    xycoords="axes fraction",
    arrowprops={"arrowstyle": "-|>", "linewidth": 3, "color": "black", "mutation_scale": 30},
    annotation_clip=False,
    gid="kids_colorbar",
)
colorbar.ax.yaxis.labelpad = 45  # keep the colorbar title clear of the arrow

kids_svg = output_svg.with_name(f"{output_svg.stem}_kids.svg")
figure.savefig(kids_svg, format="svg", bbox_inches="tight")
display(figure)
print(f"Saved {kids_svg}")